# Phase 2: verified passages, through MCP

Runs the Phase 2 graph on **the same topic as Phase 1**, so the two briefs can be compared, and keeps the record of the run:

1. the brief, rendered below and saved to `notebooks/phase_02_brief.md`;
2. the run numbers (sub-questions, pages read and skipped, findings kept and dropped, sections, retries, time);
3. the link to the run's trace in LangSmith;
4. at the end, **the phase record**: the problems met while building Phase 2 and what was decided.

What changed since Phase 1: the researcher reaches the web only through the project's own MCP server, `search-mcp`. For each sub-question it searches, reads up to three pages, and asks P-02 for the passages that answer the question; code keeps a passage only if it occurs word for word on its page.

**Before running:** start `search-mcp` in its own terminal and leave it running:

```powershell
uv run python -m mcp_servers.search_mcp.server
```

Needs real keys in `.env`. One run costs up to 14 model calls and 6 Tavily searches, and takes about three minutes (sub-questions run one after another until Phase 3).

Outputs are stripped from this file when it is committed (`nbstripout`); the brief file is the committed record. Concept: `tools-through-mcp`.

In [ ]:
# Work from the project root, wherever Jupyter started: settings.py reads .env
# from the working folder, and the graph package is imported from the root.
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT.name}")

In [ ]:
import logging
import re
import socket
import time
from urllib.parse import urlsplit
from uuid import uuid4

from IPython.display import Markdown, display
from langchain_core.tracers.langchain import wait_for_all_tracers

from graph.llm import ModelGateway
from graph.mcp_client import START_SERVER_COMMAND, McpToolbox
from graph.nodes.researcher import PAGES_PER_QUESTION, SEARCH_RESULTS_PER_QUESTION
from graph.prompts import planner as planner_prompt
from graph.prompts import web_findings as web_findings_prompt
from graph.prompts import writer as writer_prompts
from graph.report import cited_finding_ids, render_brief
from graph.supervisor import build_graph, research_run_config
from settings import settings


class RunLogRecorder(logging.Handler):
    """Keeps the graph's log messages, so skips, drops, and retries can be counted."""

    def __init__(self) -> None:
        super().__init__(level=logging.INFO)
        self.messages: list[str] = []

    def emit(self, record: logging.LogRecord) -> None:
        self.messages.append(record.getMessage())


# Every module under graph/ logs to a child of the "graph" logger.
graph_logger = logging.getLogger("graph")
graph_logger.setLevel(logging.INFO)
for old_recorder in [h for h in graph_logger.handlers if isinstance(h, RunLogRecorder)]:
    graph_logger.removeHandler(old_recorder)  # re-running this cell must not count twice
run_log = RunLogRecorder()
graph_logger.addHandler(run_log)

In [ ]:
# CONCEPT: mcp-client. Check that search-mcp is listening before spending any
# model call; McpToolbox.connect would also refuse, with the same command.
server_address = urlsplit(settings.vre_search_mcp_url)
try:
    with socket.create_connection((server_address.hostname, server_address.port or 80), 1.0):
        print(f"search-mcp is running at {settings.vre_search_mcp_url}")
except OSError:
    raise RuntimeError(
        f"search-mcp is not running; start it with: {START_SERVER_COMMAND}"
    ) from None

In [ ]:
TOPIC = "Due diligence on Symbotic: business model, customers, and main risks"

# CONCEPT: tracing-from-day-one. The run id is chosen before the run, so the
# trace can be looked up afterwards (cell below).
run_id = uuid4()
run_log.messages.clear()
started_at = time.perf_counter()

# CONCEPT: tools-through-mcp. The notebook owns the toolbox's one connection;
# the graph only uses it (D-25, D-37). The connection closes when the block ends.
async with McpToolbox.connect(settings.vre_search_mcp_url) as toolbox:
    graph = build_graph(ModelGateway(), toolbox)
    final_state = await graph.ainvoke(
        {"topic": TOPIC, "document_ids": []},
        config=research_run_config(TOPIC, run_id),
    )

elapsed_seconds = time.perf_counter() - started_at
print(f"Done in {elapsed_seconds:.1f} s")

In [ ]:
brief_markdown = render_brief(TOPIC, final_state["draft"], final_state["findings"])
display(Markdown(brief_markdown))

In [ ]:
def count_messages(fragment: str) -> int:
    return sum(fragment in message for message in run_log.messages)


def dropped_by_reason() -> dict[str, int]:
    reasons = re.findall(r"dropped a finding \(([^)]+)\)", "\n".join(run_log.messages))
    return {reason: reasons.count(reason) for reason in sorted(set(reasons))}


findings = final_state["findings"]
kept_ids = {finding.id for finding in findings if finding.found}
run_numbers = {
    "sub-questions": len(final_state["sub_questions"]),
    "pages skipped": count_messages(": skipped "),
    "searches failed": count_messages(": search failed"),
    "findings kept": len(kept_ids),
    "sub-questions with no evidence": sum(not finding.found for finding in findings),
    "findings dropped by the checks": sum(dropped_by_reason().values()),
    "sources cited": len(cited_finding_ids(final_state["draft"]) & kept_ids),
    "sections": len(final_state["draft"]),
    "unknown markers removed": count_messages("removed unknown markers"),
    "schema retries": count_messages("did not fit"),
    "transient retries": count_messages("transient error"),
    "seconds": round(elapsed_seconds, 1),
}
for measure, value in run_numbers.items():
    print(f"{measure:<34}{value}")
print(f"{'dropped, by reason':<34}{dropped_by_reason() or 'none'}")

In [ ]:
# CONCEPT: tracing-from-day-one. LangSmith uploads runs from a background
# thread; wait until everything is sent, then find this run by its id.
wait_for_all_tracers()

if settings.langsmith_tracing:
    from langsmith import Client

    langsmith_client = Client()
    for lookup_attempt in range(1, 6):
        try:
            traced_run = langsmith_client.read_run(run_id)
            trace_url = langsmith_client.get_run_url(
                run=traced_run, project_name=settings.langsmith_project
            )
            print(f"Trace: {trace_url}")
            break
        except Exception:  # a new run can take a few seconds to appear
            time.sleep(2 * lookup_attempt)
    else:
        print(f"Trace not found yet: search project {settings.langsmith_project} for {run_id}")
else:
    print("Tracing is off (LANGSMITH_TRACING=false)")

In [ ]:
# The committed record of the run. Only the brief and its provenance are
# written: never settings, keys, or LangSmith ids.
prompt_versions = (
    f"P-01 {planner_prompt.VERSION}, P-02 {web_findings_prompt.VERSION}, "
    f"P-03 {writer_prompts.OUTLINE_VERSION}, P-04 {writer_prompts.SECTION_VERSION}"
)
provenance_lines = [
    "> - Generated by `notebooks/phase_02.ipynb`, Phase 2. Same topic as `phase_01_brief.md`.",
    f"> - Prompts: {prompt_versions}.",
    f"> - Models: `{settings.vre_chat_model}` (planner, extraction), "
    f"`{settings.vre_writer_model}` (writer).",
    f"> - Search and page reading: `search-mcp` (MCP), {SEARCH_RESULTS_PER_QUESTION} results "
    f"and up to {PAGES_PER_QUESTION} pages per sub-question.",
    f"> - Run: {run_numbers['sub-questions']} sub-questions, {run_numbers['findings kept']} "
    f"findings kept, {run_numbers['findings dropped by the checks']} dropped by the checks, "
    f"{run_numbers['sections']} sections, {run_numbers['seconds']} s.",
    "> - Phase 2: every passage below occurs word for word on the page it cites, checked in "
    "code. Whether each sentence is supported by its passage is checked from phase 5.",
]
title_line, brief_body = brief_markdown.split("\n", 1)
brief_file_text = "\n".join([title_line, "", *provenance_lines, brief_body])

brief_path = PROJECT_ROOT / "notebooks" / "phase_02_brief.md"
brief_path.write_text(brief_file_text, encoding="utf-8")
print(f"Wrote {brief_path.relative_to(PROJECT_ROOT)} ({len(brief_file_text.splitlines())} lines)")

## Phase 1 and Phase 2, same topic

Measured on the live runs that closed each phase:

| | Phase 1 | Phase 2 |
| --- | --- | --- |
| Where a passage comes from | The snippet the search engine returned | A passage P-02 chose from a page it read, **checked to occur word for word on that page** |
| Findings kept | 18 (every snippet) | 13 |
| Off-topic evidence | An unrelated company, filing boilerplate, a page titled "VP sells shares" | None seen |
| Sub-questions without evidence | 0, because a snippet always counts as "found" | 0 |
| How the graph reaches the web | Tavily, called directly | Only through `search-mcp`, checked by a test that fails on any search import in `graph/` |
| Time | 43.6 s | 172.8 s (parallel researchers arrive in Phase 3) |

Fewer findings, each of which is real text from the page it cites and answers its question.

## Problems we hit and what we decided

The phase record. One entry per problem met while building Phase 2: what happened, what caused it (or what a check showed when the first guess was wrong), the options, and the decision. Decision numbers refer to the project's decision log.

### 1. An empty page looked like a readable one

**What happened.** While writing `fetch_page`, a test page with nothing but a menu came back with the text `Menu`. The text-extraction library falls back to any text it can find when a page has no article.
**Decision.** Main text below a minimum length counts as "no readable text", and the tool says so instead of returning a scrap. **Lesson:** a library's fallback can look like success, so test the empty case too.

### 2. A video page slipped past that minimum

**What happened.** The first live run counted a video page as read, with 160 characters of leftovers (a title, a line of description, footer links). The minimum was 80.
**Options.** Raise the minimum; block video sites by name; leave it.
**Decision.** Minimum raised to 300 characters. It works on any site, and real articles were all above 2,500. A test with 135 characters of leftovers was checked to fail at 80 and pass at 300.

### 3. A refusal that was not our fault

**What happened.** Wikipedia answered `403 Forbidden` in the first live run. The obvious explanation was the request's User-Agent.
**The check before any change.** The same request was repeated four ways, including exactly what the server sent: all returned `200`. The refusal was temporary.
**Decision.** No code change for the 403; the page was skipped as designed. Separately, the User-Agent now names the public repository, as automated readers are asked to. The same page was read normally in later runs. **Lesson:** check the likely explanation before acting on it.

### 4. The planned MCP client library could not be installed (D-34)

**What happened.** The design named `langchain-mcp-adapters` for the graph's MCP client. Tried before planning around it, its newest release required an older version of the `mcp` library and stopped our server from starting; the previous release crashed on import.
**Options.** FastMCP's own client; downgrading FastMCP; separate environments for server and graph.
**Decision.** FastMCP's own client, behind a small `Toolbox` interface, with one tracing decorator per tool call to replace what the adapters would have traced. Same library on both sides of the protocol; switching later changes one file.

### 5. Half the extracted passages were not real copies (D-36, D-39)

**What happened.** The model asked to copy passages word for word returned text the check could not find on the page: 3 of 6 in the first check. Two causes looked obvious from the dropped text, and both were wrong. A diagnostic that shows where each copy breaks found the real ones: the model **joined text from two or three places** into one "quote", and **corrected a typo** on the page.
**First decision (D-36).** Keep the prompt and tune it later against an evaluation set, because six findings were too few to judge a change.
**What changed our mind.** The first full graph run lost **every** finding on 2 of 6 sub-questions, so the brief had four sections instead of six. The failure was costing whole parts of the brief.
**Second decision (D-39).** Prompt v2 now: one continuous stretch of one to three sentences, copied character for character with mistakes included; "give two findings" instead of joining; a worked example of a wrong and a right passage. Rescuing part of a bad copy in code was rejected, because the answer could rest on the part that was not copied.

| Full graph run, same topic | v1 | v2 |
| --- | --- | --- |
| Findings kept | 5 of 11 | 13 of 17 |
| Sub-questions with no evidence | 2 of 6 | 0 of 6 |
| Sections | 4 | 6 |

**What remains.** The model still sometimes corrects a typo; the check still drops that passage. It costs one finding, not a sub-question.

### 6. A network outage exposed two gaps (D-40)

**What happened.** Two runs stopped with "search-mcp stopped answering". The cause, read from the run's own output, was the machine's name resolution failing: the server was fine, Tavily was unreachable.
**Gap 1.** The server's search had no time limit of its own, so the client's limit fired first and blamed the server. **Decision:** every search has a 15 s limit in the server, which reports "search timed out" itself.
**Gap 2.** With that fixed, a dead network would have turned every sub-question into "no evidence" and still produced a brief. **Decision:** if every search fails, the run stops with a clear error. One failed search still costs only its own sub-question.
**Good news from the same event:** the run stopped instead of writing an empty brief, exactly as designed.

### 7. The Phase 1 notebook was retired (D-37)

The direct search client was deleted so the graph can only reach the web through MCP, and a test fails if any file in `graph/` imports a search, page-reading, or PDF library. `phase_01.ipynb` used that client, so it can no longer run. It stays as the record of Phase 1, with its brief, `phase_01_brief.md`, as the "before" in the comparison above.